In [1]:
import os
import shutil
import pandas as pd
from tqdm import tqdm

In [ ]:
def organize_ISIC2019_by_class(csv_path, image_dir, output_dir):
    class_cols = ['MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC', 'UNK']

    df = pd.read_csv(csv_path)

    # Create label column from one-hot encoded class columns
    df['label'] = df[class_cols].values.argmax(axis=1)
    df['label_name'] = df['label'].map(lambda x: class_cols[x])

    # Construct full filename with .jpg extension
    df['filename'] = df['image'].astype(str) + '.jpg'

    # Create output folders for each class
    for class_name in class_cols:
        os.makedirs(os.path.join(output_dir, class_name), exist_ok=True)

    print(f"📁 Organizing images into: {output_dir}")

    # Copy images to class folders
    for _, row in tqdm(df.iterrows(), total=len(df), desc="📦 Copying images"):
        src_path = os.path.join(image_dir, row['filename'])
        dst_path = os.path.join(output_dir, row['label_name'], row['filename'])

        if os.path.exists(src_path):
            shutil.copy2(src_path, dst_path)
        else:
            print(f"⚠️ Missing file: {row['filename']}")

    print("✅ All images organized successfully.")

In [ ]:
organize_ISIC2019_by_class(
    csv_path='ISIC_2019_Training_GroundTruth.csv',
    image_dir='ISIC_2019_Training_Input',
    output_dir='ISIC_2019_organized/Train'
)

📁 Organizing images into: ISIC_2019_organized/Train


📦 Copying images: 100%|██████████| 25331/25331 [08:53<00:00, 47.49it/s]

✅ All images organized successfully.


In [ ]:
organize_ISIC2019_by_class(
    csv_path='ISIC_2019_Test_GroundTruth.csv',
    image_dir='ISIC_2019_Test_Input',
    output_dir='ISIC_2019_organized/Test'
)

📁 Organizing images into: ISIC_2019_organized/Test


📦 Copying images: 100%|██████████| 8238/8238 [01:53<00:00, 72.28it/s] 

✅ All images organized successfully.


In [2]:
def organize_ham10000_images(csv_path, image_dir, output_dir):
    # Read metadata
    df = pd.read_csv(csv_path)

    # Extract diagnosis labels
    class_labels = df['dx'].unique()

    # Create folders for each class
    for label in class_labels:
        os.makedirs(os.path.join(output_dir, label), exist_ok=True)

    # Construct full filename
    df['filename'] = df['image_id'].astype(str) + '.jpg'

    print(f"📁 Organizing HAM10000 images into: {output_dir}")

    # Copy images to class-labeled directories
    for _, row in tqdm(df.iterrows(), total=len(df), desc="📦 Copying images"):
        src_path = os.path.join(image_dir, row['filename'])
        dst_path = os.path.join(output_dir, row['dx'], row['filename'])

        if os.path.exists(src_path):
            shutil.copy2(src_path, dst_path)
        else:
            #print(f"⚠️ Missing file: {row['filename']}")
            pass

    print("✅ All images organized successfully.")

In [3]:
organize_ham10000_images(
    csv_path='ha\HAM10000_metadata.csv',
    image_dir='ha\HAM10000_images_part_1',
    output_dir='HAM10000_organized'
)

📁 Organizing HAM10000 images into: HAM10000_organized


📦 Copying images: 100%|██████████| 10015/10015 [01:31<00:00, 109.45it/s]

✅ All images organized successfully.


In [4]:
organize_ham10000_images(
    csv_path='ha\HAM10000_metadata.csv',
    image_dir='ha\HAM10000_images_part_2',
    output_dir='HAM10000_organized'
)

📁 Organizing HAM10000 images into: HAM10000_organized


📦 Copying images: 100%|██████████| 10015/10015 [01:36<00:00, 103.27it/s]

✅ All images organized successfully.


In [7]:
def organize_MILK10K_by_class(csv_path, image_dir, output_dir):
    # Class labels in dataset
    class_cols = ['AKIEC','BCC','BEN_OTH','BKL','DF','INF','MAL_OTH','MEL','NV','SCCKA','VASC']

    # Load CSV
    df = pd.read_csv(csv_path)

    # Create label column from one-hot encoded class columns
    df['label'] = df[class_cols].values.argmax(axis=1)
    df['label_name'] = df['label'].map(lambda x: class_cols[x])

    # Create output folders for each class
    for class_name in class_cols:
        os.makedirs(os.path.join(output_dir, class_name), exist_ok=True)

    print(f"📁 Organizing images into: {output_dir}")

    # Copy images from lesion_id folders to their class folders
    for _, row in tqdm(df.iterrows(), total=len(df), desc="📦 Copying images"):
        lesion_folder = os.path.join(image_dir, str(row['lesion_id']))

        if not os.path.exists(lesion_folder):
            print(f"⚠️ Missing folder: {lesion_folder}")
            continue

        # Collect all images inside lesion_id folder
        images = [f for f in os.listdir(lesion_folder) if f.lower().endswith((".jpg",".png",".jpeg"))]

        if not images:
            print(f"⚠️ No images found in {lesion_folder}")
            continue

        for img_file in images:
            src_path = os.path.join(lesion_folder, img_file)
            dst_path = os.path.join(output_dir, row['label_name'], img_file)

            # Avoid overwriting if duplicate names exist
            if os.path.exists(dst_path):
                base, ext = os.path.splitext(img_file)
                i = 1
                while os.path.exists(dst_path):
                    dst_path = os.path.join(output_dir, row['label_name'], f"{base}_{i}{ext}")
                    i += 1

            shutil.copy2(src_path, dst_path)

    print("✅ All images organized successfully.")

In [8]:
organize_MILK10K_by_class(
    csv_path=r'MILK10K\MILK10k_Training_GroundTruth.csv',
    image_dir=r'MILK10K\MILK10k_Training_Input',
    output_dir=r'MILK10K\MILK10k_Organized'
)

📁 Organizing images into: MILK10K\MILK10k_Organized


📦 Copying images: 100%|██████████| 5240/5240 [01:15<00:00, 69.53it/s] 

✅ All images organized successfully.
